# env

In [1]:
import os, sys, logging, json, pandas as pd, numpy as np
from glob import glob
from tqdm import tqdm
from multiprocessing import cpu_count
print(cpu_count())

import warnings
warnings.filterwarnings("ignore")
DATADIR="../"

sys.path.insert(0, "../")
os.environ['PYTHONPATH']="../"


def create_model_soup(model_names, output_name=None):
    import torch
    from safetensors.torch import load_file, save_file
    
    model_names = sorted(model_names.split())
    print(model_names)
    ckpt_dirs = [sorted(glob(f"../data/{model_name}/checkpoint-*"), key=lambda x: int(x.split('-')[-1]))[-1] for model_name in model_names]
    if output_name is None:
        output_name = re.sub('KF0', 'ms_KF0', model_names[0])
    output_dir = f"../data/{output_name}"
    os.makedirs(output_dir)
    !cp  ../data/{model_names[0]}/args.json {output_dir}
    
    output_ckpt = f"{output_dir}/checkpoint-1000000"
    !cp -r {ckpt_dirs[0]} {output_ckpt}
    print(output_ckpt)
    
    soup_state_dict = load_file(f"{ckpt_dirs[0]}/model.safetensors")
    
    # Convert to float32 for precision during averaging if they are in fp16/bf16
    for key in soup_state_dict:
        soup_state_dict[key] = soup_state_dict[key].to(torch.float32)

    # Accumulate weights from the remaining models
    num_models = len(ckpt_dirs)
    for i in tqdm(range(1, num_models)):
        print(f"Adding {ckpt_dirs[i]} to the soup...")
        current_model = load_file(f"{ckpt_dirs[i]}/model.safetensors")
        
        for key in soup_state_dict:
            # Add the weights (ensure they are on the same device and precision)
            soup_state_dict[key] += current_model[key].to(torch.float32)
        
        # Free memory
        del current_model

    # Divide by the number of models to get the average
    print("Averaging weights...")
    for key in soup_state_dict:
        soup_state_dict[key] = (soup_state_dict[key] / num_models).to(torch.bfloat16) # Or torch.float16

    # Save the result
    print(f"Saving soup to {output_ckpt}...")
    save_file(soup_state_dict, f"{output_ckpt}/model.safetensors")
    print("Done!")

24


# preprocess

In [ ]:
!python preprocess.py -m to_16k_noise 

# stage1

## train CSRW_qw3asr_1d7b_d23

In [ ]:
import util
LOGDIR=f"{DATADIR}/logs"
!mkdir -p {LOGDIR}
LOGFILE=f"{LOGDIR}/log{util.timestamp()}.log"
print(LOGFILE)


MODELNAME="CSRW_qw3asr_1d7b_d23"
DS="csrw"
DSCLS="QWASRDataset"
VDSCLS="QWASRDataset"
TDSCLS="QWASRDataset"
RESTORE=""
BACKBONE="Qwen/Qwen3-ASR-1.7B"


DATASEED=33234566
SEED=98020302
KN=5
KFIDS="0"
MAXSEC=45





!nvidia-smi


!TRANSFORMERS_OFFLINE=1 python -u sft.py -model_name {MODELNAME} -ds "{DS}" -ds_cls "{DSCLS}" -val_ds_cls "{VDSCLS}" -test_ds_cls "{TDSCLS}" \
-kn {KN}  -kfids "{KFIDS}" -backbone "{BACKBONE}" -data_seed {DATASEED} -seed {SEED} -eval_delay 0 -do_train -do_val -n_dl_worker 8  -verbose 64 \
-eval_strategy epoch -save_strategy epoch -epochs 2 -es 1 -n_keep_save 1 -torch_dtype bfloat16 -mixed_precision bf16   \
-lr 2e-5 -lr_warmup_ratio 0.01 -weight_decay 0.01 -bs 2 -gas 32 -vbs 4 -groupfy_col "child_id" -stratify_col "age_bucket"  \
-max_sec {MAXSEC} -sr 16000 -use_tb  -mask_time_prob 0.05 -am_bn_noise 0.2 -nv -compile_model \
2>&1 | tee "{LOGFILE}"




2026-04-01 02:51:37,265 - INFO - __main__ -   backbone: Qwen/Qwen3-ASR-1.7B, kfid: 0
🚨 `loss_kwargs` is part of Qwen3ASRThinkerForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/tom/github/csrw/qwen_asr/core/transformers_backend/modeling_qwen3_asr.py.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 91.06it/s]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-04-01 02:51:39,498 - INFO - __main__ -   pad token id:151643, None, padding side:right
2026-04-01 02:51:39,500 - INFO - __main__ -   num of params (2038.05248, 0.0)
2026-04-01 02:51:40,497 - INFO - util -   num of csrw is 350112
2026-04-01 02:51:41,354 - INFO - __main__ -   ds cls:<class '__main__.QWASRDataset'>
2026-04-01 02:51:41,355 - INFO - util -   preprocess start
2026-04-01 02:51:41,455 - INFO - dat

In [ ]:

os.chdir(os.path.join(WORKDIR, PROJECT))

import util
LOGDIR=f"{DATADIR}/logs"
!mkdir -p {LOGDIR}
LOGFILE=f"{LOGDIR}/log{util.timestamp()}.log"
print(LOGFILE)


MODELNAME="CSRW_qw3asr_1d7b_d22"
DS="csrw"
DSCLS="QWASRDataset"
VDSCLS="QWASRDataset"
TDSCLS="QWASRDataset"
RESTORE=""
BACKBONE="Qwen/Qwen3-ASR-1.7B"


DATASEED=55549888
SEED=33322901
KN=5
KFIDS="0"
MAXSEC=45





!nvidia-smi


!TRANSFORMERS_OFFLINE=1 python -u sft.py -model_name {MODELNAME} -ds "{DS}" -ds_cls "{DSCLS}" -val_ds_cls "{VDSCLS}" -test_ds_cls "{TDSCLS}" \
-kn {KN}  -kfids "{KFIDS}" -backbone "{BACKBONE}" -data_seed {DATASEED} -seed {SEED} -eval_delay 0 -do_train -do_val -n_dl_worker 8  -verbose 64 \
-eval_strategy epoch -save_strategy epoch -epochs 2 -es 1 -n_keep_save 1 -torch_dtype bfloat16 -mixed_precision bf16   \
-lr 2e-5 -lr_warmup_ratio 0.01 -weight_decay 0.01 -bs 2 -gas 32 -vbs 4 -groupfy_col "child_id" -stratify_col "age_bucket"  \
-max_sec {MAXSEC} -sr 16000 -use_tb  -mask_time_prob 0.05 -am_bn_noise 0.2 -nv -compile_model \
2>&1 | tee "{LOGFILE}"




2026-03-31 07:49:30,275 - INFO - __main__ -   backbone: Qwen/Qwen3-ASR-1.7B, kfid: 0
🚨 `loss_kwargs` is part of Qwen3ASRThinkerForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/tom/github/csrw/qwen_asr/core/transformers_backend/modeling_qwen3_asr.py.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.44s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-03-31 07:49:53,776 - INFO - __main__ -   pad token id:151643, None, padding side:right
2026-03-31 07:49:53,810 - INFO - __main__ -   num of params (2038.05248, 0.0)
2026-03-31 07:49:54,824 - INFO - util -   num of csrw is 350112
2026-03-31 07:49:55,683 - INFO - __main__ -   ds cls:<class '__main__.QWASRDataset'>
2026-03-31 07:49:55,683 - INFO - util -   preprocess start
2026-03-31 07:49:55,784 - INFO - dat

In [ ]:

os.chdir(os.path.join(WORKDIR, PROJECT))

import util
LOGDIR=f"{DATADIR}/logs"
!mkdir -p {LOGDIR}
LOGFILE=f"{LOGDIR}/log{util.timestamp()}.log"
print(LOGFILE)


MODELNAME="CSRW_qw3asr_1d7b_d21"
DS="csrw"
DSCLS="QWASRDataset"
VDSCLS="QWASRDataset"
TDSCLS="QWASRDataset"
RESTORE=""
BACKBONE="Qwen/Qwen3-ASR-1.7B"


DATASEED=44334989
SEED=23409827
KN=5
KFIDS="0"
MAXSEC=45





!nvidia-smi


!TRANSFORMERS_OFFLINE=1 python -u sft.py -model_name {MODELNAME} -ds "{DS}" -ds_cls "{DSCLS}" -val_ds_cls "{VDSCLS}" -test_ds_cls "{TDSCLS}" \
-kn {KN}  -kfids "{KFIDS}" -backbone "{BACKBONE}" -data_seed {DATASEED} -seed {SEED} -eval_delay 0 -do_train -do_val -n_dl_worker 8  -verbose 64 \
-eval_strategy epoch -save_strategy epoch -epochs 2 -es 1 -n_keep_save 1 -torch_dtype bfloat16 -mixed_precision bf16   \
-lr 2e-5 -lr_warmup_ratio 0.01 -weight_decay 0.01 -bs 2 -gas 32 -vbs 4 -groupfy_col "child_id" -stratify_col "age_bucket"  \
-max_sec {MAXSEC} -sr 16000 -use_tb  -mask_time_prob 0.05 -am_bn_noise 0.2 -nv -compile_model \
2>&1 | tee "{LOGFILE}"




2026-03-30 22:31:50,554 - INFO - __main__ -   backbone: Qwen/Qwen3-ASR-1.7B, kfid: 0
🚨 `loss_kwargs` is part of Qwen3ASRThinkerForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/tom/github/csrw/qwen_asr/core/transformers_backend/modeling_qwen3_asr.py.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.09s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-03-30 22:32:40,061 - INFO - __main__ -   pad token id:151643, None, padding side:right
2026-03-30 22:32:40,062 - INFO - __main__ -   num of params (2038.05248, 0.0)
2026-03-30 22:32:41,082 - INFO - util -   num of csrw is 350112
2026-03-30 22:32:41,948 - INFO - __main__ -   ds cls:<class '__main__.QWASRDataset'>
2026-03-30 22:32:41,948 - INFO - util -   preprocess start
2026-03-30 22:32:42,048 - INFO - dat

In [ ]:

os.chdir(os.path.join(WORKDIR, PROJECT))

import util
LOGDIR=f"{DATADIR}/logs"
!mkdir -p {LOGDIR}
LOGFILE=f"{LOGDIR}/log{util.timestamp()}.log"
print(LOGFILE)


MODELNAME="CSRW_qw3asr_1d7b_d20"
DS="csrw"
DSCLS="QWASRDataset"
VDSCLS="QWASRDataset"
TDSCLS="QWASRDataset"
RESTORE=""
BACKBONE="Qwen/Qwen3-ASR-1.7B"


DATASEED=58475019
SEED=45837264
KN=5
KFIDS="0"
MAXSEC=45





!nvidia-smi


!TRANSFORMERS_OFFLINE=1 python -u sft.py -model_name {MODELNAME} -ds "{DS}" -ds_cls "{DSCLS}" -val_ds_cls "{VDSCLS}" -test_ds_cls "{TDSCLS}" \
-kn {KN}  -kfids "{KFIDS}" -backbone "{BACKBONE}" -data_seed {DATASEED} -seed {SEED} -eval_delay 0 -do_train -do_val -n_dl_worker 8  -verbose 64 \
-eval_strategy epoch -save_strategy epoch -epochs 2 -es 1 -n_keep_save 1 -torch_dtype bfloat16 -mixed_precision bf16   \
-lr 2e-5 -lr_warmup_ratio 0.01 -weight_decay 0.01 -bs 2 -gas 32 -vbs 4 -groupfy_col "child_id" -stratify_col "age_bucket"  \
-max_sec {MAXSEC} -sr 16000 -use_tb  -mask_time_prob 0.05 -am_bn_noise 0.2 -nv -compile_model \
2>&1 | tee "{LOGFILE}"




2026-03-30 05:23:30,366 - INFO - __main__ -   backbone: Qwen/Qwen3-ASR-1.7B, kfid: 0
🚨 `loss_kwargs` is part of Qwen3ASRThinkerForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/tom/github/csrw/qwen_asr/core/transformers_backend/modeling_qwen3_asr.py.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.45s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-03-30 05:23:54,121 - INFO - __main__ -   pad token id:151643, None, padding side:right
2026-03-30 05:23:54,122 - INFO - __main__ -   num of params (2038.05248, 0.0)
2026-03-30 05:23:55,212 - INFO - util -   num of csrw is 350112
2026-03-30 05:23:56,092 - INFO - __main__ -   ds cls:<class '__main__.QWASRDataset'>
2026-03-30 05:23:56,093 - INFO - util -   preprocess start
2026-03-30 05:23:56,194 - INFO - dat

In [ ]:

os.chdir(os.path.join(WORKDIR, PROJECT))

import util
LOGDIR=f"{DATADIR}/logs"
!mkdir -p {LOGDIR}
LOGFILE=f"{LOGDIR}/log{util.timestamp()}.log"
print(LOGFILE)


MODELNAME="CSRW_qw3asr_1d7b_d19"
DS="csrw"
DSCLS="QWASRDataset"
VDSCLS="QWASRDataset"
TDSCLS="QWASRDataset"
RESTORE=""
BACKBONE="Qwen/Qwen3-ASR-1.7B"


DATASEED=85746309
SEED=87463849
KN=5
KFIDS="0"
MAXSEC=45





!nvidia-smi


!TRANSFORMERS_OFFLINE=1 python -u sft.py -model_name {MODELNAME} -ds "{DS}" -ds_cls "{DSCLS}" -val_ds_cls "{VDSCLS}" -test_ds_cls "{TDSCLS}" \
-kn {KN}  -kfids "{KFIDS}" -backbone "{BACKBONE}" -data_seed {DATASEED} -seed {SEED} -eval_delay 0 -do_train -do_val -n_dl_worker 8  -verbose 64 \
-eval_strategy epoch -save_strategy epoch -epochs 2 -es 1 -n_keep_save 1 -torch_dtype bfloat16 -mixed_precision bf16   \
-lr 2e-5 -lr_warmup_ratio 0.01 -weight_decay 0.01 -bs 2 -gas 32 -vbs 4 -groupfy_col "child_id" -stratify_col "age_bucket"  \
-max_sec {MAXSEC} -sr 16000 -use_tb  -mask_time_prob 0.05 -am_bn_noise 0.2 -nv -compile_model \
2>&1 | tee "{LOGFILE}"




2026-03-23 07:12:58,011 - INFO - __main__ -   backbone: Qwen/Qwen3-ASR-1.7B, kfid: 0
🚨 `loss_kwargs` is part of Qwen3ASRThinkerForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/tom/github/csrw/qwen_asr/core/transformers_backend/modeling_qwen3_asr.py.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.45s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-03-23 07:13:21,911 - INFO - __main__ -   pad token id:151643, None, padding side:right
2026-03-23 07:13:21,933 - INFO - __main__ -   num of params (2038.05248, 0.0)
2026-03-23 07:13:22,998 - INFO - util -   num of csrw is 350112
2026-03-23 07:13:23,750 - INFO - __main__ -   ds cls:<class '__main__.QWASRDataset'>
2026-03-23 07:13:23,750 - INFO - util -   preprocess start
2026-03-23 07:13:23,844 - INFO - dat

## create model soup

In [ ]:
model_names = "CSRW_qw3asr_1d7b_d19_KF0 CSRW_qw3asr_1d7b_d20_KF0 CSRW_qw3asr_1d7b_d21_KF0 CSRW_qw3asr_1d7b_d22_KF0 CSRW_qw3asr_1d7b_d23_KF0"
create_model_soup(model_names)

## create semi1

In [ ]:

os.chdir(os.path.join(WORKDIR, PROJECT))




MODELNAME="CSRW_qw3asr_1d7b_d19_ms"
DS="csrw3"



KN=5
KFIDS="0"
MAXSEC=45







!python -u sft.py -model_name "{MODELNAME}"  -kn {KN} -kfids "{KFIDS}" -m predict -ds {DS} -test_ds_cls "QWASRIterDataset" -do_test -data_type test  \
-restore -vbs 8  -n_dl_worker 8   -torch_dtype bfloat16  \
-temp 1 -n_beam 5  -max_gen_len 144  -max_sec {MAXSEC} -no_merge -min_sec 2 


In [ ]:
import util
args = util.parser.parse_args([])
args.dataset = 'csrw3'
args.max_sec=45
csrw3 = util.load_data(args)
csrw_semi1 = util.load_dump('../data/CSRW_qw3asr_1d7b_d19_ms_KF0/pred_test.dump')
print(len(csrw3))
print(len(csrw_semi1))
print(csrw_semi1.utterance_id.unique().shape)
csrw_semi1 = csrw3.merge(csrw_semi1, on='utterance_id')

print(len(csrw_semi1))
csrw_semi1['orthographic_text'] = csrw_semi1.pred
print(len(csrw_semi1))
csrw_semi1 = csrw_semi1[csrw_semi1.orthographic_text.apply(len)>0]
print(len(csrw_semi1))
csrw_semi1['src'] = 'semi1'
csrw_semi1['ID'] = range(len(csrw_semi1))
util.dump(csrw_semi1, '../data/csrw_semi1.dump')

# stage 2

## train CSRW_qw3asr_1d7b_d24

In [ ]:

os.chdir(os.path.join(WORKDIR, PROJECT))

import util
LOGDIR=f"{DATADIR}/logs"
!mkdir -p {LOGDIR}
LOGFILE=f"{LOGDIR}/log{util.timestamp()}.log"
print(LOGFILE)


MODELNAME="CSRW_qw3asr_1d7b_d24"
DS="csrw"
DSCLS="QWASRDataset"
VDSCLS="QWASRDataset"
TDSCLS="QWASRDataset"
RESTORE=""
BACKBONE="Qwen/Qwen3-ASR-1.7B"


DATASEED=56857648
SEED=32349084
KN=5
KFIDS="0 1 2 3 4"
MAXSEC=45





!nvidia-smi


!TRANSFORMERS_OFFLINE=1 python -u sft.py -model_name {MODELNAME} -ds "{DS}" -ds_cls "{DSCLS}" -val_ds_cls "{VDSCLS}" -test_ds_cls "{TDSCLS}" \
-kn {KN}  -kfids "{KFIDS}" -backbone "{BACKBONE}" -data_seed {DATASEED} -seed {SEED} -eval_delay 0 -do_train -do_val -n_dl_worker 8  -verbose 64 \
-eval_strategy epoch -save_strategy epoch -epochs 2 -es 1 -n_keep_save 1 -torch_dtype bfloat16 -mixed_precision bf16   \
-lr 2e-5 -lr_warmup_ratio 0.01 -weight_decay 0.01 -bs 2 -gas 32 -vbs 4 -groupfy_col "child_id" -stratify_col "age_bucket"  \
-max_sec {MAXSEC} -sr 16000 -use_tb  -mask_time_prob 0.05 -am_bn_noise 0.2 -nv -compile_model -use_semi -semi_fpath "../data/csrw_semi1.dump" \
2>&1 | tee "{LOGFILE}"




2026-04-01 11:14:42,435 - INFO - __main__ -   backbone: Qwen/Qwen3-ASR-1.7B, kfid: 0
🚨 `loss_kwargs` is part of Qwen3ASRThinkerForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/tom/github/csrw/qwen_asr/core/transformers_backend/modeling_qwen3_asr.py.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 91.67it/s]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-04-01 11:14:44,635 - INFO - __main__ -   pad token id:151643, None, padding side:right
2026-04-01 11:14:44,636 - INFO - __main__ -   num of params (2038.05248, 0.0)
2026-04-01 11:14:45,661 - INFO - util -   num of csrw is 350112
/home/tom/anaconda3/envs/csrw/lib/python3.12/site-packages/sklearn/model_selection/_split.py:1037: UserWarning: The least populated class in y has only 2 members, which is less tha

## train CSRW_qw3asr_1d7b_d25

In [ ]:

os.chdir(os.path.join(WORKDIR, PROJECT))

import util
LOGDIR=f"{DATADIR}/logs"
!mkdir -p {LOGDIR}
LOGFILE=f"{LOGDIR}/log{util.timestamp()}.log"
print(LOGFILE)


MODELNAME="CSRW_qw3asr_1d7b_d25"
DS="csrw"
DSCLS="QWASRDataset"
VDSCLS="QWASRDataset"
TDSCLS="QWASRDataset"
RESTORE=""
BACKBONE="Qwen/Qwen3-ASR-1.7B"


DATASEED=56857648
SEED=32349084
KN=10
KFIDS="0 1 2 3 4 5 6 7"
MAXSEC=45





!nvidia-smi


!TRANSFORMERS_OFFLINE=1 python -u sft.py -model_name {MODELNAME} -ds "{DS}" -ds_cls "{DSCLS}" -val_ds_cls "{VDSCLS}" -test_ds_cls "{TDSCLS}" \
-kn {KN}  -kfids "{KFIDS}" -backbone "{BACKBONE}" -data_seed {DATASEED} -seed {SEED} -eval_delay 0 -do_train -do_val -n_dl_worker 8  -verbose 64 \
-eval_strategy epoch -save_strategy epoch -epochs 2 -es 1 -n_keep_save 1 -torch_dtype bfloat16 -mixed_precision bf16   \
-lr 2e-5 -lr_warmup_ratio 0.01 -weight_decay 0.01 -bs 2 -gas 32 -vbs 4 -groupfy_col "child_id" -stratify_col "age_bucket"  \
-max_sec {MAXSEC} -sr 16000 -use_tb  -mask_time_prob 0.05 -am_bn_noise 0 -nv -compile_model -use_semi -semi_fpath "../data/csrw_semi1.dump" \
2>&1 | tee "{LOGFILE}"




2026-04-04 02:38:24,729 - INFO - __main__ -   backbone: Qwen/Qwen3-ASR-1.7B, kfid: 0
🚨 `loss_kwargs` is part of Qwen3ASRThinkerForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/tom/github/csrw/qwen_asr/core/transformers_backend/modeling_qwen3_asr.py.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.46s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-04-04 02:38:48,230 - INFO - __main__ -   pad token id:151643, None, padding side:right
2026-04-04 02:38:48,231 - INFO - __main__ -   num of params (2038.05248, 0.0)
2026-04-04 02:38:49,311 - INFO - util -   num of csrw is 350112
/home/tom/anaconda3/envs/csrw/lib/python3.12/site-packages/sklearn/model_selection/_split.py:1037: UserWarning: The least populated class in y has only 2 members, which is less tha

# create final soup

In [ ]:
model_names = "CSRW_qw3asr_1d7b_d19_KF0 CSRW_qw3asr_1d7b_d20_KF0 CSRW_qw3asr_1d7b_d21_KF0 CSRW_qw3asr_1d7b_d22_KF0 CSRW_qw3asr_1d7b_d23_KF0 CSRW_qw3asr_1d7b_d24_KF0 CSRW_qw3asr_1d7b_d24_KF1 CSRW_qw3asr_1d7b_d24_KF2 CSRW_qw3asr_1d7b_d24_KF3 CSRW_qw3asr_1d7b_d24_KF4 \
CSRW_qw3asr_1d7b_d25_KF0 CSRW_qw3asr_1d7b_d25_KF1 CSRW_qw3asr_1d7b_d25_KF2 CSRW_qw3asr_1d7b_d25_KF3 CSRW_qw3asr_1d7b_d25_KF4 CSRW_qw3asr_1d7b_d25_KF5 \
CSRW_qw3asr_1d7b_d25_KF6 CSRW_qw3asr_1d7b_d25_KF7"
create_model_soup(model_names, output_name="CSRW_qw3asr_1d7b_d25_ms_KF0")